# SR benchmark on an NVIDIA T4 (Colab / Kaggle)

The plan reports **cost on an NVIDIA T4**: seconds per image and peak GPU memory.
This notebook sets the project up on a T4 runtime and runs `scripts/benchmark_runtime.py`
(the frozen settings in `configs/frozen.yaml` are used). Optionally it runs the whole
pipeline on the GPU.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

Get the code: clone your team repository (set `REPO_URL`), or upload `cv-project.zip` to the session and unzip it.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<team>/cv-project.git"
import os
if REPO_URL:
    !git clone -q $REPO_URL cv-project
elif os.path.exists("cv-project.zip"):
    !unzip -q cv-project.zip
%cd cv-project

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null   # only needed for the OCR metric
!pip -q install -e .
!bash scripts/setup_third_party.sh

Download data and weights, build the HR/LR sets (~10 min, mostly the DIV2K archive).

In [ ]:
!python scripts/download.py
!python scripts/prepare_data.py

## Cost on the T4
Batch size 1, 128×128 → 512×512, 3 warm-up images, 30 timed images per method. Writes `results/runtime/Tesla_T4.csv`.

In [ ]:
!python scripts/benchmark_runtime.py --device cuda --n 30

In [ ]:
import pandas as pd
rt = pd.read_csv("results/runtime/Tesla_T4.csv")
rt.groupby("method").agg(sec_mean=("seconds", "mean"), sec_median=("seconds", "median"), peak_mb=("peak_mb", "max")).round(3)

Download the CSV and commit it to `results/runtime/` so `scripts/analyze.py` includes it.

In [ ]:
from google.colab import files   # on Kaggle: use the output panel instead
files.download("results/runtime/Tesla_T4.csv")

## Optional: the full pipeline on the GPU
About 1 hour on a T4. Note that ResShift is stochastic and CUDA draws different random
numbers than MPS/CPU, so its outputs will not be bit-identical to the reported run.

In [ ]:
# !python scripts/run_sr.py --split dev && python scripts/run_sr.py --split test
# !python scripts/evaluate.py --split test && python scripts/seed_experiment.py && python scripts/analyze.py